# Titans Board v2.0 — Colab クイックスタート

CFO / CLO / CEO / 監査役 の AI 取締役会を Colab 上で動かすセットアップです。

**推奨ランタイム**: `T4 GPU`（メニュー → ランタイム → ランタイムのタイプを変更）  
CPUでも動きますが Qwen3-4B の応答が遅くなります（1レスポンス30秒〜）。

---
## Step 0 — ランタイム確認

In [ ]:
import subprocess, sys
!nvidia-smi 2>/dev/null || echo 'CPU ランタイム（GPUなし）'
!free -h | head -2
!df -h / | tail -1

## Step 1 — Ollama インストール & サーバー起動

In [ ]:
# Ollama 本体インストール
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, os

# バックグラウンドでサーバー起動
_ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)  # 起動待ち

# 起動確認
import urllib.request
try:
    urllib.request.urlopen("http://localhost:11434", timeout=5)
    print("Ollama: OK")
except Exception as e:
    print(f"Ollama: NG — {e}")

## Step 2 — モデル取得

| モデル | サイズ | 用途 |
|--------|--------|------|
| `qwen3:4b` | 約 2.4 GB | 取締役会の推論（必須） |
| `nomic-embed-text` | 約 274 MB | 意味検索（任意・推奨） |

In [ ]:
# 必須: 推論モデル（約2.4GB・数分かかります）
!ollama pull qwen3:4b

In [ ]:
# 任意: 埋め込みモデル（意味検索の精度向上・約274MB）
# 入れない場合はオフラインの hashing embedder が使われます
!ollama pull nomic-embed-text

## Step 3 — リポジトリ取得 & 依存インストール

In [ ]:
# リポジトリクローン（開発ブランチを直接取得 — PRやマージは不要）
!git clone -b claude/titans-board-v2-design-pb81x7 https://github.com/nori1234/Stock_Tracker-657.git titans-board
%cd titans-board

In [ ]:
# Python パッケージインストール
!pip install -q -r requirements.txt

In [ ]:
# nomic-embed-text を入れた場合は embedder を ollama に切り替える
# （入れていない場合はこのセルをスキップ）
import re, pathlib
cfg = pathlib.Path("config.yaml")
cfg.write_text(cfg.read_text().replace(
    "embedder: hashing", "embedder: ollama"
))
print("embedder: ollama に変更しました")

## Step 4 — 接続確認

In [ ]:
!python main.py --health-check

`Status: OK` が出れば準備完了です。

---
## Step 5 — 知識ベース取り込み（任意）

In [ ]:
# サンプル知識ファイルを取り込む
!python main.py --ingest ./knowledge

## Step 6 — 長期記憶に方針・禁止事項を登録（任意）

In [ ]:
!python main.py --remember "ギャンブル・アダルト関連事業への参入禁止" --category 禁止事項
!python main.py --remember "3年以内の黒字化を全事業の必須条件とする" --category 経営方針
!python main.py --memories

---
## Step 7 — 取締役会を開催する

CFO → CLO → CEO草稿 → 監査役 → CEO最終 の順で審議が進みます。  
T4 GPU で 1〜3 分、CPU では 5〜15 分程度かかります。

In [ ]:
AGENDA = "新規事業として、AIを活用した医療診断支援サービスを日本市場で展開したい。初期投資5億円、3年でのROI達成が目標。取締役会の判断を仰ぎたい。"

!python main.py "{AGENDA}"

In [ ]:
# 保存されたレポートを確認
import glob, json, pathlib
files = sorted(glob.glob("outputs/meeting_*.json"))
if files:
    latest = json.loads(pathlib.Path(files[-1]).read_text())
    print(f"保存先: {files[-1]}")
    print("\n=== CEO最終判断 ===")
    print(latest.get("ceo_final", "")[:1000])

---
## オプション: Anthropic API を使う場合

Ollama の代わりに Claude を使いたい場合（GPU 不要・応答が速い）。

In [ ]:
import os
from google.colab import userdata

# Colab シークレット（左メニュー 🔑）に ANTHROPIC_API_KEY を登録しておく
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# config.yaml を Anthropic に切り替え
cfg = pathlib.Path("config.yaml")
text = cfg.read_text()
text = text.replace("provider: ollama", "provider: anthropic")
text = text.replace("model: qwen3:4b", "model: claude-haiku-4-5-20251001")
cfg.write_text(text)
print("provider: anthropic に切り替えました")

!python main.py --health-check

---
## トラブルシュート

| 症状 | 対処 |
|------|------|
| `Status: FAIL` | Step 1 のサーバー起動セルを再実行 |
| `model not found` | `!ollama pull qwen3:4b` を再実行 |
| セッション切れ後に起動しない | Step 1〜2 のセルを再実行（Colab はセッションごとにリセット） |
| GPU メモリ不足 | `qwen3:4b` は約 3GB VRAM。T4（16GB）で余裕あり |
| 応答が遅い | CPU ランタイムの場合。T4 GPU に切り替えると数倍速くなる |